# Gold export to CSV + ZIP

This notebook exports Delta tables from the Gold lakehouse to CSV under `Files/sources` and then creates one ZIP per table under `Files/sources_zip`.

Use case: static dataset backup for GitHub portability.

In [ ]:
from pyspark.sql import Row

from pyspark.sql.utils import AnalysisException

import os

import re

import tempfile

import zipfile

import shutil

from datetime import datetime



# Source catalog/schema

LAKEHOUSE_NAME = "Gold"

SCHEMA_NAME = "dbo"



# Output folders inside the attached lakehouse

CSV_BASE = "Files/sources"

ZIP_BASE = "Files/sources_zip"



# Ontology AnalyzeLoanOnV3 authoritative dependencies from DataBindings

TABLE_SPECS = [

    {"logical_name": "dim_loan", "required": True, "candidates": ["dim_loan"]},

    {"logical_name": "dim_prev_application", "required": True, "candidates": ["dim_prev_application"]},

    {"logical_name": "dim_relative_month", "required": True, "candidates": ["dim_relative_month"]},

    {

        "logical_name": "dim_dpd_bucket_v2",

        "required": True,

        "candidates": ["dim_dpd_bucket_v2", "dim_dpd_bucketV2", "dim_dpdV2", "dim_dpd_v2", "dim_dpd_bucket"],

        "contains_tokens": ["dim", "dpd", "v2"],

    },

    {

        "logical_name": "fact_pos_cash_monthly_balance_v2",

        "required": True,

        "candidates": ["fact_pos_cash_monthly_balance_v2", "fact_pos_cash_monthly_balanceV2", "fact_credit_card_monthly_balanceV2", "fact_credit_card_monthly_balance_v2", "fact_credit_card_monthly_balance"],

        "contains_tokens": ["fact", "monthly", "balance", "v2"],

    },

]



OVERWRITE = True

ONE_FILE_PER_TABLE = True

COUNT_ROWS = True

ZIP_COMPRESSION_LEVEL = 9



print(f"CSV base: {CSV_BASE}")

print(f"ZIP base: {ZIP_BASE}")

print(f"Table specs: {len(TABLE_SPECS)}")


In [ ]:
def fq_table(table_name: str) -> str:

    return f"{LAKEHOUSE_NAME}.{SCHEMA_NAME}.{table_name}"



def csv_folder(logical_name: str) -> str:

    return f"{CSV_BASE}/{logical_name}"



def zip_file_path(logical_name: str) -> str:

    return f"{ZIP_BASE}/{logical_name}.zip"



def normalize_name(value: str) -> str:

    return re.sub(r"[^a-z0-9]", "", value.lower())



def list_available_tables():

    df = spark.sql(f"SHOW TABLES IN {LAKEHOUSE_NAME}.{SCHEMA_NAME}")

    return [row["tableName"] for row in df.select("tableName").collect()]



def resolve_table_name(spec: dict, available_tables):

    normalized_map = {normalize_name(t): t for t in available_tables}



    for candidate in spec.get("candidates", []):

        normalized_candidate = normalize_name(candidate)

        if normalized_candidate in normalized_map:

            return normalized_map[normalized_candidate]



    tokens = [normalize_name(token) for token in spec.get("contains_tokens", []) if token]

    if tokens:

        for table_name in available_tables:

            normalized_table = normalize_name(table_name)

            if all(token in normalized_table for token in tokens):

                return table_name



    return None



def resolve_tables():

    available = list_available_tables()

    resolved = []

    missing_required = []



    for spec in TABLE_SPECS:

        found = resolve_table_name(spec, available)

        resolved.append({

            "logical_name": spec["logical_name"],

            "table_name": found,

            "required": spec.get("required", False),

            "candidates": ";".join(spec.get("candidates", [])),

        })

        if spec.get("required", False) and not found:

            missing_required.append(spec["logical_name"])



    if missing_required:

        raise ValueError(

            "Missing required V2-aware tables: "

            + ", ".join(missing_required)

            + ". Available tables: "

            + ", ".join(sorted(available))

        )



    return resolved



def safe_rm(path: str, recursive: bool = True):

    try:

        notebookutils.fs.rm(path, recursive)

    except Exception:

        pass



def ensure_dir(path: str):

    try:

        notebookutils.fs.mkdirs(path)

    except Exception:

        pass



def export_table_to_csv(item: dict):

    logical_name = item["logical_name"]

    table_name = item["table_name"]

    src = fq_table(table_name)

    dst = csv_folder(logical_name)



    try:

        df = spark.read.table(src)

        row_count = df.count() if COUNT_ROWS else None



        if OVERWRITE:

            safe_rm(dst, True)



        writer_df = df.coalesce(1) if ONE_FILE_PER_TABLE else df

        writer_df.write.mode("overwrite").option("header", "true").csv(dst)



        return {

            "logical_name": logical_name,

            "table_name": table_name,

            "source_table": src,

            "csv_folder": dst,

            "row_count": row_count,

            "exported": True,

            "error": None,

        }

    except AnalysisException as ex:

        return {

            "logical_name": logical_name,

            "table_name": table_name,

            "source_table": src,

            "csv_folder": dst,

            "row_count": None,

            "exported": False,

            "error": str(ex),

        }

    except Exception as ex:

        return {

            "logical_name": logical_name,

            "table_name": table_name,

            "source_table": src,

            "csv_folder": dst,

            "row_count": None,

            "exported": False,

            "error": str(ex),

        }



def zip_csv_folder(item: dict):

    logical_name = item["logical_name"]

    src_folder = csv_folder(logical_name)

    dst_zip = zip_file_path(logical_name)



    try:

        entries = notebookutils.fs.ls(src_folder)

        csv_files = [e.path for e in entries if e.path.lower().endswith(".csv")]



        if len(csv_files) == 0:

            return {

                "logical_name": logical_name,

                "table_name": item["table_name"],

                "zip_path": dst_zip,

                "compressed": False,

                "files_in_zip": 0,

                "zip_bytes": None,

                "error": "No CSV files found in source folder",

            }



        ensure_dir(ZIP_BASE)

        if OVERWRITE:

            safe_rm(dst_zip, False)



        with tempfile.TemporaryDirectory() as tmp:

            local_files = []



            for src_csv in csv_files:

                file_name = src_csv.rstrip("/").split("/")[-1]

                local_csv = os.path.join(tmp, file_name)

                notebookutils.fs.cp(src_csv, f"file:{local_csv}")

                local_files.append(local_csv)



            local_zip = os.path.join(tmp, f"{logical_name}.zip")

            with zipfile.ZipFile(local_zip, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=ZIP_COMPRESSION_LEVEL) as zf:

                for local_csv in local_files:

                    zf.write(local_csv, arcname=os.path.basename(local_csv))



            zip_size = os.path.getsize(local_zip)

            notebookutils.fs.cp(f"file:{local_zip}", dst_zip)



        return {

            "logical_name": logical_name,

            "table_name": item["table_name"],

            "zip_path": dst_zip,

            "compressed": True,

            "files_in_zip": len(csv_files),

            "zip_bytes": zip_size,

            "error": None,

        }

    except Exception as ex:

        return {

            "logical_name": logical_name,

            "table_name": item["table_name"],

            "zip_path": dst_zip,

            "compressed": False,

            "files_in_zip": 0,

            "zip_bytes": None,

            "error": str(ex),

        }



RESOLVED_TABLES = resolve_tables()

resolved_df = spark.createDataFrame([Row(**r) for r in RESOLVED_TABLES])

display(resolved_df.orderBy("logical_name"))


In [ ]:
export_results = [export_table_to_csv(item) for item in RESOLVED_TABLES]

export_df = spark.createDataFrame([Row(**r) for r in export_results])

display(export_df.orderBy("logical_name"))



if export_df.filter("exported = false").count() > 0:

    raise ValueError("Some tables failed during CSV export. Check export_df for details.")


In [ ]:
zip_results = [zip_csv_folder(item) for item in RESOLVED_TABLES]

zip_df = spark.createDataFrame([Row(**r) for r in zip_results])

display(zip_df.orderBy("logical_name"))



failed_zip = zip_df.filter("compressed = false").count()

if failed_zip > 0:

    raise ValueError("Some tables failed during ZIP compression. Check zip_df for details.")



summary = spark.createDataFrame([Row(

    generated_at_utc=datetime.utcnow().isoformat(),

    table_count=len(RESOLVED_TABLES),

    csv_base=CSV_BASE,

    zip_base=ZIP_BASE,

    total_zip_mb=round((zip_df.selectExpr("sum(zip_bytes) as b").collect()[0]["b"] or 0) / (1024*1024), 3)

)])

display(summary)


In [ ]:
MANIFEST_BASE = "Files/sources_manifest"
ensure_dir(MANIFEST_BASE)

export_df.coalesce(1).write.mode("overwrite").option("header", "true").csv(f"{MANIFEST_BASE}/export_results")
zip_df.coalesce(1).write.mode("overwrite").option("header", "true").csv(f"{MANIFEST_BASE}/zip_results")

print("Done.")
print(f"CSV folders: {CSV_BASE}")
print(f"ZIP files: {ZIP_BASE}")
print(f"Manifest: {MANIFEST_BASE}")